# MVB-01 — Dataset Acquisition & Profiling

**Project:** P001-MVB — Explainable and Responsible AI for Differentiating Bacterial and Viral Meningitis: A Clinical Decision Support System
**Stage:** 01_MVB_Data-Acquisition
**Dataset:** Meningitis Classification (Kaggle, ChanTest)
**Date:** 21 July 2026

This notebook acquires and profiles the raw dataset before any cleaning or transformation occurs. The goal is to understand structure, quality, and initial clinical signal — establishing a factual baseline for the decisions made in Stage 02 (Data Preparation).

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Set working directory
import os
os.chdir('/content/drive/MyDrive/P001-MVB/01_MVB_Data-Acquisition')

In [ ]:

# Only pandas is needed for profiling at this stage
import pandas as pd

In [ ]:

# Load the untouched raw file from the acquisition folder
df = pd.read_csv("MVB-01-dataset-raw/meningitis.csv")

## 1. Overview

In [ ]:

# First look at the data
df.head()

,Patient_ID,Age,Gender,WBC_Count,Protein_Level,Glucose_Level,Pathogen_Present,Diagnosis,Outcome,Hemoglobin,WBC_Blood_Count,Platelets,CRP_Level,Risk_Level
0,1,66,Female,14450,179,19,Yes,Bacterial,Recovered,3,13912,119405,45,High Risk
1,2,94,Male,13470,122,104,No,Bacterial,Recovered,16,6845,213495,44,Moderate Risk
2,3,23,Male,8921,20,66,No,Viral,Recovered,18,4049,217301,5,Low Risk
3,4,53,Female,16200,145,16,Yes,Bacterial,Recovered,1,17731,119570,44,High Risk
4,5,47,Female,4781,43,58,No,Viral,Recovered,14,5086,259521,0,Low Risk


In [ ]:
# Dataset dimensions
print("Shape:", df.shape)

Shape: (1200, 14)


In [ ]:

# Column types and non-null counts in one view
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 14 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Patient_ID        1200 non-null   int64 
 1   Age               1200 non-null   int64 
 2   Gender            1200 non-null   object
 3   WBC_Count         1200 non-null   int64 
 4   Protein_Level     1200 non-null   int64 
 5   Glucose_Level     1200 non-null   int64 
 6   Pathogen_Present  1200 non-null   object
 7   Diagnosis         1200 non-null   object
 8   Outcome           1200 non-null   object
 9   Hemoglobin        1200 non-null   int64 
 10  WBC_Blood_Count   1200 non-null   int64 
 11  Platelets         1200 non-null   int64 
 12  CRP_Level         1200 non-null   int64 
 13  Risk_Level        1200 non-null   object
dtypes: int64(9), object(5)
memory usage: 131.4+ KB


## 2. Data Quality Assessment
Checking for missing values and duplicate records before any further profiling.

In [ ]:

# Missing values per column
df.isnull().sum()

,0
Patient_ID,0
Age,0
Gender,0
WBC_Count,0
Protein_Level,0
Glucose_Level,0
Pathogen_Present,0
Diagnosis,0
Outcome,0
Hemoglobin,0


## 3. Schema Inspection

In [ ]:
# Column names and data types
df.dtypes

,0
Patient_ID,int64
Age,int64
Gender,object
WBC_Count,int64
Protein_Level,int64
Glucose_Level,int64
Pathogen_Present,object
Diagnosis,object
Outcome,object
Hemoglobin,int64


In [ ]:
# Summary statistics for numeric columns
df.describe()

,Patient_ID,Age,WBC_Count,Protein_Level,Glucose_Level,Hemoglobin,WBC_Blood_Count,Platelets,CRP_Level
count,1200.000000,1200.00000,1200.000000,1200.000000,1200.00000,1200.000000,1200.000000,1200.000000,1200.000000
mean,600.500000,45.24750,11842.602500,101.002500,52.55750,9.653333,11720.409167,196849.476667,24.575000
std,346.554469,21.82246,5370.192798,68.895268,36.47518,5.567374,4911.127424,91320.569325,22.778061
min,1.000000,0.00000,2008.000000,2.000000,0.00000,1.000000,4022.000000,100088.000000,0.000000
25%,300.750000,29.00000,7052.750000,39.000000,23.00000,5.000000,7225.750000,123773.750000,3.000000
50%,600.500000,41.00000,11834.500000,101.000000,53.00000,8.000000,12454.500000,148116.000000,25.000000
75%,900.250000,57.00000,16434.000000,155.000000,67.00000,15.000000,16077.750000,266625.750000,40.000000
max,1200.000000,119.00000,24717.000000,299.000000,148.00000,18.000000,19991.000000,399479.000000,99.000000


## 4. Categorical Variables
Distribution of each categorical field.

In [ ]:
df["Gender"].value_counts()

,count
Gender,
Female,607
Male,593


In [ ]:
df["Pathogen_Present"].value_counts()

,count
Pathogen_Present,
Yes,631
No,569


In [ ]:
df["Outcome"].value_counts()

,count
Outcome,
Recovered,1076
Deceased,124


In [ ]:
df["Risk_Level"].value_counts()

,count
Risk_Level,
High Risk,593
Low Risk,540
Moderate Risk,67


## 5. Target Variable Assessment — `Diagnosis`

The project objective is binary classification (Bacterial vs. Viral). The raw data contains a third class, "Unknown," which will require a handling decision in Stage 02.

In [ ]:
df["Diagnosis"].value_counts()

,count
Diagnosis,
Bacterial,595
Viral,538
Unknown,67


## 6. Data Quality Checks

Investigating two flagged issues: whether `Risk_Level` overlaps with the "Unknown" `Diagnosis` class, and whether `Age` contains implausible values.

In [ ]:
# Check whether "Unknown" Diagnosis and "Moderate Risk" are the same population
# (both happened to total 67 rows)
overlap = df[(df["Diagnosis"] == "Unknown") & (df["Risk_Level"] == "Moderate Risk")]
print("Rows that are BOTH Unknown diagnosis AND Moderate Risk:", len(overlap))

Rows that are BOTH Unknown diagnosis AND Moderate Risk: 20


In [ ]:
# Full cross-tabulation to confirm Risk_Level is not a proxy for Diagnosis
pd.crosstab(df["Diagnosis"], df["Risk_Level"])

Risk_Level,High Risk,Low Risk,Moderate Risk
Diagnosis,,,
Bacterial,501,69,25
Unknown,23,24,20
Viral,69,447,22


In [ ]:
# Age plausibility — check extreme values
print("Rows with Age > 100:", (df["Age"] > 100).sum())
print("Rows with Age == 0:", (df["Age"] == 0).sum())

Rows with Age > 100: 43
Rows with Age == 0: 2


## 7. Clinical Plausibility Assessment

Comparing the three core CSF biomarkers across diagnosis classes. Bacterial meningitis is clinically expected to show higher WBC count, higher protein, and lower glucose than viral meningitis.

In [23]:
for col in ["WBC_Count", "Protein_Level", "Glucose_Level"]:
    print(f"--- {col} by Diagnosis ---")
    print(df.groupby("Diagnosis")[col].agg(["min", "max", "mean"]).round(1))
    print()

--- WBC_Count by Diagnosis ---
            min    max     mean
Diagnosis                      
Bacterial  2008  23870  15039.5
Unknown    2888  24717  14031.9
Viral      2304  24716   8034.3

--- Protein_Level by Diagnosis ---
           min  max   mean
Diagnosis                 
Bacterial   10  299  139.4
Unknown      7  299  140.2
Viral        2  297   53.7

--- Glucose_Level by Diagnosis ---
           min  max  mean
Diagnosis                
Bacterial    1  148  36.3
Unknown      0  146  73.0
Viral        3  148  68.0



## 8. Summary

- Dataset: 1,200 rows × 14 columns, no missing values, no duplicate rows or Patient_IDs.
- Target variable `Diagnosis` has three classes: Bacterial (595), Viral (538), Unknown (67) — the Unknown cases require a handling decision in Stage 02.
- `Risk_Level` is not a proxy for `Diagnosis`; the matching count of 67 was coincidental (only 20 rows overlap).
- Age = 0 (n=2) rows are complete and internally consistent — interpreted as infant cases, not data errors.
- Age > 100 (n=43) rows require plausibility review in Stage 02.
- Core CSF markers (WBC, Protein, Glucose) show clinically plausible separation between Bacterial and Viral classes, indicating meaningful diagnostic signal in the dataset.

**Next stage:** 02_MVB_Data-Preparation